#### Please note that it is better to run the imports, gird setup, initial conditions, comparing settings, define of hydro functions first and then run the calculate cells using FTCS, Lax-Friedrichs, Lax-Wendroff seperately. After running one such calcualtion cell, it is better to clear the output and run another calculation cell. 

# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Grid setup

In [ ]:
# Define your grid here. E.g. the number of (ghost) cells, the boxsize, ...
gamma = 1.4 # The value of adiabatic index
deltat = 0.0003
deltax = 0.01
q = np.empty([3, 152], dtype = float)# initialize the matrix to store the value of each cells

# Initial conditions

In [ ]:
# Define your initial conditions here.
# Initialise your state vector q (containing density, momentum and energy), e.g. as zeros with shape (3, N_CELLS)
# Use a few variables to set different ranges of the grid to the different initial conditions
for i in range(75):
    q[0,i+1] = 2.5
    q[1,i+1] = 0
    q[2,i+1] = 1.5/(gamma-1)
    q[0,150-i] = 0.125
    q[1,150-i] = 0
    q[2,150-i] = 0.1/(gamma-1)
    
# The setup of ghost cells to guarantee zero gradient with the adjacent cells
for i in range(3):
    q[i,0] = q[i,1]
    q[i,151] = q[i,150]

# Comparison settings

In [ ]:
# Load analytical solution:
# x, rho, P, u, e
q_exact_003 = np.genfromtxt('./sod_exact_003.txt',skip_header=1,delimiter=',')
q_exact_03 = np.genfromtxt('./sod_exact_03.txt',skip_header=1,delimiter=',')


# Define hydro functions

In [ ]:
# Define the functions you will need here.

def flux_function(q, gamma):
    """ Define a function to determine the fluxfunction f(q) for a state vector q.
    Remember that q should contain three different quantities. """
    # calculate the flux given a state vector q
    for i in range(151):
        if i>0 :
        # define temporary variable for simplicity
            q1 = q[0,i]
            q2 = q[1,i]
            q3 = q[2,i]
            f[0,i-1] = q2
            f[1,i-1] = ((q2*q2)/q1) + (gamma-1)*(q3-0.5*q2*q2/q1)
            f[2,i-1] = (q3+(gamma-1)*(q3-0.5*q2*q2/q1))*(q2/q1)
    # return the flux vector
    return f

def flux_function_interface(Q, gamma):
    """ Define a function to determine the fluxfunction f(Q) for a interface state Q."""

    # calculate the flux given a state vector q
    for i in range(149):
        # define temporary variable for simplicity
            q1 = Q[0,i]
            q2 = Q[1,i]
            q3 = Q[2,i]
            f[0,i] = q2
            f[1,i] = ((q2*q2)/q1) + (gamma-1)*(q3-0.5*q2*q2/q1)
            f[2,i] = (q3+(gamma-1)*(q3-0.5*q2*q2/q1))*(q2/q1)
    # return the flux vector
    return f

def update_q_with_fluxes(q, f, F, gamma, deltat, deltax):
    """ Define a function that updates the state vector q with the appropriate fluxes. """
    # calculate something
    for i in range(150):
        for j in range(3):
            if ((i>1) and (i<150)):
                q[j,i] = q[j,i] - (deltat/deltax)*(F[j,i-1]-F[j,i-2])
            elif i == 1:
                q[j,i] = q[j,i] - (deltat/deltax)*F[j,0]
            elif i == 150:
                q[j,i] = q[j,i] + (deltat/deltax)*F[j,148]
            else: # first give the unchanged value to the ghost cells, they will be given value after this loop
                q[j,i] = q[j,i]
            
    # give value to the ghost cell after the flux have been updated
    for j in range(3):
        q[j,0] = q[j,1]
        q[j,151] = q[j,150]
                

    return q
    # return something

def calculate_flux_FTCS(f, gamma):
    """ Define a function that calculates the fluxes for a state vector q using the FTCS method. """
    # calculate the flux at each interface of the cells
    for i in range(149):
        for j in range(3):
            F[j,i] = 0.5*(f[j,i]+f[j,i+1])
    # return result of the calculation of the interface flux using FTCS
    return F

def calculate_flux_Lax_Friedrichs(f, gamma, deltax, deltat, q):
    """ Define a function that calculates the fluxes for a state vector q using the Lax-Friedrichs method. """
    #calculate the flux at each interface of the cells
    F = np.empty([3, 149], dtype = float)
    for i in range(149):
        for j in range(3):
            F[j,i] = 0.5*(f[j,i]+f[j,i+1]) - 0.5*(deltax/deltat)*(q[j,i+2]-q[j,i+1]) 
    # return result of the calculatetion of the interface flux using Lax-Friedrichs
    return F

def calculate_flux_Lax_Wendroff(f, gamma, deltax, deltat, q):
    #calculate the flux at each interface of the cells
    Q = np.empty([3, 149], dtype = float)
    for i in range(149):
        for j in range(3):
            Q[j,i] = 0.5*(q[j,i+2]+q[j,i+1])-0.5*(deltat/deltax)*(f[j,i+1]-f[j,i])
    F = flux_function_interface(Q,gamma)
    return F
# Once you have finished and tested the calculate_flux_FTCS-function, make a new function where you
# update it to implement the Lax-Friedrich method. After that make another function to implement
# the Lax-Wendroff method, so that you can always compare to the old functions.
   

## Calculating the evolution: Sod shock tube

In [ ]:
# Place the actual calculations here. Make a loop (e.g. a while-loop) and update (a copy of) the state vector each
# iteration. Save it, and check for boundary conditions.
# Repeat for the different methods you will implement. Remember to make separate blocks, headlines, comments...
gamma = 1.4 # The value of adiabatic index
deltat = 0.0003
deltax = 0.01
f = np.empty([3, 150], dtype = float)
F = np.empty([3, 149], dtype = float)

for k in range(100): # advance the solution 100 times from t=0 to t= 0.03
    f = flux_function(q, gamma)  # calculate the flux using the flux function
    F = calculate_flux_FTCS(f, gamma) # calculate the flux at the interface of two cells
    q = update_q_with_fluxes(q, f, F, gamma, deltat, deltax) # calculate the updated state matrix

# make a plot of the density of a function of x
x = np.empty([152, 1], dtype = float) # make an array to store the x axis
#setup of the grid
for i in range(152):
    x[i,0] = -0.005 + 0.01*i
y =  q[0,:]
plt.plot(x, y, color='blue' ) # Function plot()
plt.title("The density of the Sod shock tube(FTCS)")                             # A title on top
plt.ylabel("Density") # A label along Y
plt.grid()            # Overlay grid lines
plt.plot(q_exact_03[:,0], q_exact_03[:,1] , color='red')
plt.show()

In [ ]:
# The following code intends to calculate the flux with the Lax-Friedrichs method and make a plot to compare
gamma = 1.4 # The value of adiabatic index
deltat = 0.0003
deltax = 0.01
f = np.empty([3, 150], dtype = float)
F = np.empty([3, 149], dtype = float)



for k in range(100): # advance the solution 100 times from t=0 to t= 0.03
    f = flux_function(q, gamma)  # calculate the flux using the flux function
    F = calculate_flux_Lax_Friedrichs(f, gamma, deltax, deltat, q) # calculate the flux at the interface of two cells
    q = update_q_with_fluxes(q, f, F, gamma, deltat, deltax) # calculate the updated state matrix

# make a plot of the density of a function of x
x = np.empty([152, 1], dtype = float) # make an array to store the x axis
#setup of the grid
for i in range(152):
    x[i,0] = -0.005 + 0.01*i
y1 =  q[0,:] #The density
y2 =  q[1,:]/q[0,:]


# Make the plots of density and velocity 
fig, axs = plt.subplots(2)
fig.suptitle('The density (above) and velocity(below) of the Sod shock tube')
axs[0].plot(x, y1, color='red' )
axs[0].plot(q_exact_03[:,0], q_exact_03[:,1] , color='blue')
axs[0].grid()
axs[1].plot(x, y2, color='red' )
axs[1].plot(q_exact_03[:,0], q_exact_03[:,3] , color='blue')
axs[1].grid()



#### The problem of Lax-Friedrichs is it severely smooths out the solution. And in the cell below, the density is obtained using the Lax-Wendroff method. The problem is the solution ocsillates strongly at the right end of the x coordinate, which makes it harder to have a good look at the whole figure. 

In [ ]:
# The following code intends to calculate the flux with the Lax-Wendroff method and make a plot to compare
gamma = 1.4 # The value of adiabatic index
deltat = 0.0003
deltax = 0.01
f = np.empty([3, 150], dtype = float)
F = np.empty([3, 149], dtype = float)


for k in range(100): # advance the solution 100 times from t=0 to t= 0.03
    f = flux_function(q, gamma)  # calculate the flux using the flux function
    F = calculate_flux_Lax_Wendroff(f, gamma, deltax, deltat, q) # calculate the flux at the interface of two cells
    q = update_q_with_fluxes(q, f, F, gamma, deltat, deltax) # calculate the updated state matrix
    

# make a plot of the density of a function of x
x = np.empty([152, 1], dtype = float) # make an array to store the x axis
#setup of the grid
for i in range(152):
    x[i,0] = -0.005 + 0.01*i
y =  q[0,:]

plt.plot(x, y, color='red' ) # Function plot()
plt.title("The density of the Sod shock tube")                             # A title on top
plt.ylabel("Density")                           # A label along Y
plt.grid()                                          # Overlay grid lines
plt.plot(q_exact_03[:,0], q_exact_03[:,1] , color='blue')
plt.show()

# Sedov blast wave: grid
### Please note that it is better to clear output to run the cells below.

In [ ]:
# Reset the grid for the second test you will run. The functions will remain the same.
x = np.empty([402, 1], dtype = float) # make an array to store the x axis
#setup of the grid
for i in range(402):
    x[i,0] = -0.00025 + 0.0005*i



# Sedov blast wave: initial conditions

In [ ]:
# Reset you intial conditions.
q = np.empty([3, 402], dtype = float)# initialize the matrix to store the value of each cells
gamma = 5/3#The value of gamma

# The following is used to setup the initial condition
for i in range(402):
    if ((i==199) or (i==200) or (i ==201) or (i ==202)):
        q[0,i] = 2
        q[1,i] = 0
        q[2,i] = 60
    elif ((i>0) and (i <199)):
        q[0,i] = 2
        q[1,i] = 0
        q[2,i] = (0.00002)/(gamma-1)
    elif ((i>202) and (i<401)):
        q[0,i] = 2
        q[1,i] = 0
        q[2,i] = (0.00002)/(gamma-1)
        
# assign value to the ghost cells after the values in other cells are given
for j in range(3):
    q[j,0] = q[j,1]
    q[j,401] = q[j,400]



## Define hydrofunctions for Sedov blast wave (Because of different number of grids, we define the hydrofunctions again)

In [ ]:
# Define the functions you will need here.

def flux_function(q, gamma):
    """ Define a function to determine the fluxfunction f(q) for a state vector q.
    Remember that q should contain three different quantities. """
    # calculate the flux given a state vector q
    for i in range(401):
        if i>0 :
        # define temporary variable for simplicity
            q1 = q[0,i]
            q2 = q[1,i]
            q3 = q[2,i]
            f[0,i-1] = q2
            f[1,i-1] = ((q2*q2)/q1) + (gamma-1)*(q3-0.5*q2*q2/q1)
            f[2,i-1] = (q3+(gamma-1)*(q3-0.5*q2*q2/q1))*(q2/q1)
    # return the flux vector
    return f

def flux_function_interface(Q, gamma):
    """ Define a function to determine the fluxfunction f(Q) for a interface state Q."""

    # calculate the flux given a state vector q
    for i in range(399):
        # define temporary variable for simplicity
            q1 = Q[0,i]
            q2 = Q[1,i]
            q3 = Q[2,i]
            f[0,i] = q2
            f[1,i] = ((q2*q2)/q1) + (gamma-1)*(q3-0.5*q2*q2/q1)
            f[2,i] = (q3+(gamma-1)*(q3-0.5*q2*q2/q1))*(q2/q1)
    # return the flux vector
    return f

def update_q_with_fluxes(q, f, F, gamma, deltat, deltax):
    """ Define a function that updates the state vector q with the appropriate fluxes. """
    # calculate something
    for i in range(400):
        for j in range(3):
            if ((i>1) and (i<400)):
                q[j,i] = q[j,i] - (deltat/deltax)*(F[j,i-1]-F[j,i-2])
            elif i == 1:
                q[j,i] = q[j,i] - (deltat/deltax)*F[j,0]
            elif i == 400:
                q[j,i] = q[j,i] + (deltat/deltax)*F[j,398]
            else: # first give the unchanged value to the ghost cells, they will be given value after this loop
                q[j,i] = q[j,i]
            
    # give value to the ghost cell after the flux have been updated
    for j in range(3):
        q[j,0] = q[j,1]
        q[j,401] = q[j,400]
                

    return q
    # return something

def calculate_flux_FTCS(f, gamma):
    """ Define a function that calculates the fluxes for a state vector q using the FTCS method. """
    # calculate the flux at each interface of the cells
    for i in range(399):
        for j in range(3):
            F[j,i] = 0.5*(f[j,i]+f[j,i+1])
    # return result of the calculation of the interface flux using FTCS
    return F

def calculate_flux_Lax_Friedrichs(f, gamma, deltax, deltat, q):
    """ Define a function that calculates the fluxes for a state vector q using the Lax-Friedrichs method. """
    #calculate the flux at each interface of the cells
    F = np.empty([3, 399], dtype = float)
    for i in range(399):
        for j in range(3):
            F[j,i] = 0.5*(f[j,i]+f[j,i+1]) - 0.5*(deltax/deltat)*(q[j,i+2]-q[j,i+1]) 
    # return result of the calculatetion of the interface flux using Lax-Friedrichs
    return F

def calculate_flux_Lax_Wendroff(f, gamma, deltax, deltat, q):
    #calculate the flux at each interface of the cells
    Q = np.empty([3, 399], dtype = float)
    for i in range(399):
        for j in range(3):
            Q[j,i] = 0.5*(q[j,i+2]+q[j,i+1])-0.5*(deltat/deltax)*(f[j,i+1]-f[j,i])
    F = flux_function_interface(Q,gamma)
    return F
# Once you have finished and tested the calculate_flux_FTCS-function, make a new function where you
# update it to implement the Lax-Friedrich method. After that make another function to implement
# the Lax-Wendroff method, so that you can always compare to the old functions.
   

## Calculating the evolution: Sedov blast wave

In [ ]:
# Place the actual calculations here. Make separate blocks for the different methods.
# Place the actual calculations here. Make a loop (e.g. a while-loop) and update (a copy of) the state vector each
# iteration. Save it, and check for boundary conditions.
# Repeat for the different methods you will implement. Remember to make separate blocks, headlines, comments...
gamma = 5/3 # The value of adiabatic index
deltat = 0.0001
deltax = 0.0005
f = np.empty([3, 400], dtype = float)
F = np.empty([3, 399], dtype = float)

for k in range(500): # advance the solution 500 times from t=0 to t= 0.05
    f = flux_function(q, gamma)  # calculate the flux using the flux function
    F = calculate_flux_Lax_Friedrichs(f, gamma, deltax, deltat, q) # calculate the flux at the interface of two cells
    q = update_q_with_fluxes(q, f, F, gamma, deltat, deltax) # calculate the updated state matrix

y =  q[0,:]
plt.plot(x, y, color='blue' ) # Function plot()
plt.title("The density of the Sedov blast wave(Lax-Friedrichs)")                             # A title on top
plt.ylabel("Density") # A label along Y
plt.grid()            # Overlay grid lines
plt.show()

In [ ]:
gamma = 5/3 # The value of adiabatic index
deltat = 0.0001
deltax = 0.0005
f = np.empty([3, 400], dtype = float)
F = np.empty([3, 399], dtype = float)

for k in range(500): # advance the solution 500 times from t=0 to t= 0.05
    f = flux_function(q, gamma)  # calculate the flux using the flux function
    F = calculate_flux_Lax_Wendroff(f, gamma, deltax, deltat, q) # calculate the flux at the interface of two cells
    q = update_q_with_fluxes(q, f, F, gamma, deltat, deltax) # calculate the updated state matrix

y =  q[0,:]
plt.plot(x, y, color='blue' ) # Function plot()
plt.title("The density of the Sedov blast wave(Lax-Wendroff)")                             # A title on top
plt.ylabel("Density") # A label along Y
plt.grid()            # Overlay grid lines
plt.show()

### When I want to evolve this problem using Lax-Wendroff the density near the maximum oscilates strongly.